# Context Composer
> Given one or more registry prefixes, return a single, **safe** JSON-LD `@context` that callers can embed directly into a document.

* the **Registry** – to fetch/derive each vocabulary’s canonical context
* the **Collision Resolver** – to decide how two contexts can coexist
* 01_cache – so repeated fetches are cheap

In [ ]:
#| default_exp vocab.composer

## 1 – imports  

In [ ]:
#| export
from __future__ import annotations

from typing import Any, Dict, List

from cogitarelink.core.debug import get_logger
from cogitarelink.vocab.registry import registry
from cogitarelink.vocab.collision import resolver, Strategy

log = get_logger("composer")

## 2 – composer implementation  

In [ ]:
#| export
class Composer:
    "Merge one or more vocabularies into a conflict-free `@context` dict."

    # --------------------------- public API ----------------------------
    def compose(self, prefixes: List[str], support_nest=False, propagate=True) -> Dict[str, Any]:
        """
        Parameters
        ----------
        prefixes : list of registry prefixes, **ordered by priority**.
                   The first prefix is treated as the primary vocabulary.
        support_nest : bool, if True, add @nest support to the context
        propagate : bool, if False, add @propagate: false to prevent context inheritance
        
        Returns
        -------
        dict – JSON-LD object ready to drop into a document:
               `{\"@context\": ...}`
        """
        if not prefixes:
            raise ValueError("`prefixes` must contain at least one prefix.")

        # start with primary vocabulary
        ctx_primary = registry[prefixes[0]].context_payload()["@context"]
        merged: Dict[str, Any] = ctx_primary

        for p in prefixes[1:]:
            next_ctx = registry[p].context_payload()["@context"]
            plan     = resolver.choose(prefixes[0], p)

            log.debug(f"merge {p} under strategy {plan.strategy}")

            if plan.strategy is Strategy.property_scoped:
                prop = plan.details.get("property", "data")
                merged[prop] = next_ctx

            elif plan.strategy is Strategy.nested_contexts:
                # outer context goes first in the array
                outer, inner = plan.details["outer"], plan.details["inner"]
                ctx_outer = registry[outer].context_payload()["@context"]
                ctx_inner = registry[inner].context_payload()["@context"]
                merged = {"@context": [ctx_outer, ctx_inner]}

            elif plan.strategy is Strategy.graph_partition:
                # simplistic: just keep contexts in an array; the real graph
                # separation happens later in GraphManager.
                merged = {"@context": [merged, next_ctx]}

            elif plan.strategy is Strategy.context_versioning:
                # honour requested version; for now we just append
                merged = {"@context": [merged, next_ctx]}

            else:   # Strategy.separate_graphs (default)
                merged = {"@context": [merged, next_ctx]}

        # final shape must be `{\"@context\": ...}`
        if "@context" not in merged:
            merged = {"@context": merged}
            
        # Add JSON-LD 1.1 features if requested
        ctx = merged["@context"]
        
        # Add propagation control
        if not propagate:
            if isinstance(ctx, dict):
                ctx["@propagate"] = False
            elif isinstance(ctx, list) and len(ctx) > 0 and isinstance(ctx[0], dict):
                ctx[0]["@propagate"] = False
                
        # Add @nest support
        if support_nest and isinstance(ctx, dict):
            # This adds '@nest' to the context vocabulary
            ctx["@nest"] = "@nest"

        return merged


# module-level singleton for convenience
composer = Composer()

## Tests

In [ ]:
#| hide
from fastcore.test import test_eq

# 3.1 basic single-vocab compose
c1 = composer.compose(["schema"])
test_eq(isinstance(c1, dict), True)
test_eq("@context" in c1, True)

# 3.2 VC + EPCIS should apply property_scoped (credentialSubject)
c2 = composer.compose(["vc", "epcis"])
inner = c2["@context"]
# after property_scoped we expect a key `credentialSubject`
test_eq(isinstance(inner, dict), True)
test_eq("credentialSubject" in inner, True)


In [ ]:
#| hide
# Test nested_contexts strategy
from unittest.mock import Mock
orig_choose = resolver.choose
resolver.choose = lambda p1, p2: Mock(strategy=Strategy.nested_contexts, details={"outer": p1, "inner": p2})
c3 = composer.compose(["schema", "vc"])
test_eq(isinstance(c3["@context"], list), True)
resolver.choose = orig_choose

# Test graph_partition strategy
orig_choose = resolver.choose
resolver.choose = lambda p1, p2: Mock(strategy=Strategy.graph_partition, details={})
c4 = composer.compose(["schema", "vc"])
test_eq(isinstance(c4["@context"], list), True)
resolver.choose = orig_choose

# Test context_versioning strategy
orig_choose = resolver.choose
resolver.choose = lambda p1, p2: Mock(strategy=Strategy.context_versioning, details={})
c5 = composer.compose(["schema", "vc"])
test_eq(isinstance(c5["@context"], list), True)
resolver.choose = orig_choose

# Test separate_graphs strategy (default)
orig_choose = resolver.choose
resolver.choose = lambda p1, p2: Mock(strategy=Strategy.separate_graphs, details={})
c6 = composer.compose(["schema", "vc"])
test_eq(isinstance(c6["@context"], list), True)
resolver.choose = orig_choose

# Test empty prefixes raises ValueError
try: composer.compose([]); assert False
except ValueError: pass

# Test 3+ vocabularies
c7 = composer.compose(["schema", "vc", "epcis"])
test_eq("@context" in c7, True)


In [ ]:
#| hide
# Test new JSON-LD 1.1 features

# Test @propagate
ctx_no_propagate = composer.compose(["schema"], propagate=False)
assert "@propagate" in ctx_no_propagate["@context"]
assert ctx_no_propagate["@context"]["@propagate"] is False

# Test @nest support
ctx_with_nest = composer.compose(["schema"], support_nest=True)
assert "@nest" in ctx_with_nest["@context"]
assert ctx_with_nest["@context"]["@nest"] == "@nest"

# Test both features together
ctx_with_both = composer.compose(["schema"], support_nest=True, propagate=False)
assert "@nest" in ctx_with_both["@context"]
assert "@propagate" in ctx_with_both["@context"]
assert ctx_with_both["@context"]["@propagate"] is False

## Usage

In [ ]:
import json

ctx = composer.compose(["vc", "epcis"])
ctx_json = json.dumps(ctx, indent=2)
lines = ctx_json.split('\n')
print('\n'.join(lines[:20]) + '\n...')


{
  "@context": {
    "@protected": true,
    "id": "@id",
    "type": "@type",
    "description": "https://schema.org/description",
    "digestMultibase": {
      "@id": "https://w3id.org/security#digestMultibase",
      "@type": "https://w3id.org/security#multibase"
    },
    "digestSRI": {
      "@id": "https://www.w3.org/2018/credentials#digestSRI",
      "@type": "https://www.w3.org/2018/credentials#sriString"
    },
    "mediaType": {
      "@id": "https://schema.org/encodingFormat"
    },
    "name": "https://schema.org/name",
    "VerifiableCredential": {
      "@id": "https://www.w3.org/2018/credentials#VerifiableCredential",
...


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()